In [3]:
"""
Week 1, Day 3 — Prompt Engineering Baseline
Legal Domain Evaluation — Llama 3.1 8B via Groq

Objective: Establish a reproducible, metric-driven baseline before LoRA fine-tuning.

3 Prompt Strategies:
  A — Zero-Shot Direct: Minimal instruction, no examples
  B — Zero-Shot Role: Persona (senior corporate attorney) + structure constraints
  C — Few-Shot: 2 in-domain examples + target question

Outputs:
  - baseline_raw_outputs.csv: All 45 model outputs for manual scoring
  - baseline_summary.json: ROUGE-L scores per strategy + Gap A → C
  - Console: Per-strategy ROUGE-L, per-category breakdown, latency, token usage

Part of the AI Engineer Accelerated Plan — AWS MLA-C01 Track
"""

import os
from dotenv import load_dotenv
from groq import Groq

# ═══════════════════════════════════════════════════════════════
# 1. Configuration
# ═══════════════════════════════════════════════════════════════

load_dotenv()

# Groq client — get your free API key at https://console.groq.com/keys
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Model: Llama 3.1 8B on Groq's LPU hardware
MODEL_ID = "llama-3.1-8b-instant"

# Inference parameters — low temperature for factual consistency
TEMPERATURE = 0.1
MAX_TOKENS = 256

print(f"Model: {MODEL_ID}")
print(f"Temperature: {TEMPERATURE}")
print(f"Max tokens: {MAX_TOKENS}")
print()

Model: llama-3.1-8b-instant
Temperature: 0.1
Max tokens: 256



In [4]:
# ═══════════════════════════════════════════════════════════════
# 2. Dataset — 15 Legal Questions Across 3 Categories
# ═══════════════════════════════════════════════════════════════

DATASET = [
    # ── Contracts (6 questions) ──
    {
        "id": "c01",
        "category": "Contracts",
        "question": "What are the essential elements of a valid contract?",
        "expected": "A valid contract generally requires offer, acceptance, consideration, legal capacity, and a lawful purpose. Both parties must mutually assent to the terms."
    },
    {
        "id": "c02",
        "category": "Contracts",
        "question": "Is a verbal contract enforceable in court?",
        "expected": "Verbal contracts can be enforceable unless the agreement falls under the Statute of Frauds, which requires certain contracts to be in writing."
    },
    {
        "id": "c03",
        "category": "Contracts",
        "question": "What happens when one party breaches a contract?",
        "expected": "The non-breaching party may seek remedies such as damages, specific performance, or rescission depending on the contract and jurisdiction."
    },
    {
        "id": "c04",
        "category": "Contracts",
        "question": "What is indemnification in a service agreement?",
        "expected": "Indemnification is a contractual obligation to compensate another party for losses or damages arising from specified events or claims."
    },
    {
        "id": "c05",
        "category": "Contracts",
        "question": "Should an employment contract include a non-compete clause?",
        "expected": "Non-compete clauses are common but enforceability varies by jurisdiction. They must be reasonable in scope, duration, and geography to be upheld."
    },
    {
        "id": "c06",
        "category": "Contracts",
        "question": "What is a limitation of liability clause?",
        "expected": "It caps the amount one party must pay for damages arising from the contract, often excluding liabilities for gross negligence, willful misconduct, or IP infringement."
    },
    # ── NDAs (4 questions) ──
    {
        "id": "n01",
        "category": "NDAs",
        "question": "What is an NDA and when should it be used?",
        "expected": "A Non-Disclosure Agreement protects confidential information. Use it before sharing trade secrets, business plans, or proprietary data with third parties."
    },
    {
        "id": "n02",
        "category": "NDAs",
        "question": "Can an NDA be enforced if the recipient discloses information?",
        "expected": "Yes, if the NDA is well-drafted and reasonable, the disclosing party can seek injunctive relief and monetary damages for unauthorized disclosure."
    },
    {
        "id": "n03",
        "category": "NDAs",
        "question": "How long does an NDA typically last?",
        "expected": "NDAs often last for 3 to 5 years, but some obligations, such as those covering trade secrets, can survive indefinitely or until the information becomes public."
    },
    {
        "id": "n04",
        "category": "NDAs",
        "question": "What are key terms to include in a data processing agreement?",
        "expected": "Include roles of parties, data categories, processing purposes, security measures, subprocessor governance, breach notification, and audit rights."
    },
    # ── Intellectual Property (5 questions) ──
    {
        "id": "ip01",
        "category": "Intellectual Property",
        "question": "What is the difference between a patent and a trade secret?",
        "expected": "A patent protects an invention publicly for a limited time in exchange for disclosure, while a trade secret remains confidential for as long as it provides competitive value."
    },
    {
        "id": "ip02",
        "category": "Intellectual Property",
        "question": "How do I register a copyright for my software?",
        "expected": "Copyright arises automatically upon creation, but you can register it with the U.S. Copyright Office to obtain public record and the ability to sue for statutory damages."
    },
    {
        "id": "ip03",
        "category": "Intellectual Property",
        "question": "What constitutes trademark infringement?",
        "expected": "Trademark infringement occurs when an unauthorized party uses a mark that is likely to confuse consumers about the source of goods or services."
    },
    {
        "id": "ip04",
        "category": "Intellectual Property",
        "question": "Can I use open-source code in a commercial product?",
        "expected": "It depends on the license. Permissive licenses like MIT and Apache 2.0 usually allow commercial use, while GPL-style licenses may require sharing derivative source code."
    },
    {
        "id": "ip05",
        "category": "Intellectual Property",
        "question": "How can a company protect its branding internationally?",
        "expected": "Companies can file trademarks in target jurisdictions or use the Madrid Protocol to seek protection in multiple countries through a single application."
    },
]

print(f"Dataset loaded: {len(DATASET)} questions")
print(f"Categories: {set(d['category'] for d in DATASET)}")
print()

Dataset loaded: 15 questions
Categories: {'NDAs', 'Intellectual Property', 'Contracts'}



In [5]:
# ═══════════════════════════════════════════════════════════════
# 3. Prompt Templates — 3 Strategies
# ═══════════════════════════════════════════════════════════════


def strategy_a(question: str) -> str:
    """Zero-Shot Direct — minimal instruction, no examples."""
    return f"""You are a legal assistant specializing in corporate law. 
        Answer the following question concisely and accurately.
        Question: {question}
        Answer:"""


def strategy_b(question: str) -> str:
    """Zero-Shot with Role — persona + structure constraints."""
    return f"""You are a senior corporate attorney at a top-tier law firm.
        Provide a precise, technically accurate answer to the following legal question. 
        Structure your answer in 1-3 sentences.
        Question: {question}
        Answer:"""


def strategy_c(question: str) -> str:
    """Few-Shot — 2 in-domain examples + target question."""
    return f"""You are a legal assistant specializing in corporate law.
        Answer legal questions concisely and accurately.
        
        Example 1:
        Q: What is consideration in contract law?
        A: Consideration is something of value exchanged between parties to a contract,
        typically money, goods, services, or a promise to refrain from an action.
        
        Example 2:
        Q: Can an employer terminate an at-will employee without cause?
        A: Under at-will employment, either party may terminate the relationship at any time
        without cause, unless a contract specifies otherwise or termination would violate
        anti-discrimination laws.
        
        Now answer the following:
        Q: {question}
        A:"""


STRATEGIES = {
    "A — Zero-Shot Direct": strategy_a,
    "B — Zero-Shot Role": strategy_b,
    "C — Few-Shot": strategy_c,
}

print("3 prompt strategies defined:")
for name in STRATEGIES:
    print(f"  {name}")
print()

3 prompt strategies defined:
  A — Zero-Shot Direct
  B — Zero-Shot Role
  C — Few-Shot



In [6]:
import time

# ═══════════════════════════════════════════════════════════════
# 4. Inference Engine
# ═══════════════════════════════════════════════════════════════

def query_groq(prompt: str) -> dict:
    """Send a prompt to Groq and return the response + metadata."""
    start = time.time()

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )

    latency = time.time() - start

    return {
        "output": response.choices[0].message.content,
        "latency_s": round(latency, 2),
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens,
    }

In [8]:
# ═══════════════════════════════════════════════════════════════
# 5. Run Full Baseline — 45 Inferences
# ═══════════════════════════════════════════════════════════════

results = []
total = len(DATASET) * len(STRATEGIES)
completed = 0

print(f"Running {total} inferences across {len(DATASET)} questions × {len(STRATEGIES)} strategies...")
print("=" * 80)

for item in DATASET:
    row = {
        "id": item["id"],
        "category": item["category"],
        "question": item["question"],
        "expected": item["expected"],
    }

    for strategy_name, prompt_fn in STRATEGIES.items():
        prompt = prompt_fn(item["question"])
        result = query_groq(prompt)

        row[f"{strategy_name}_output"] = result["output"]
        row[f"{strategy_name}_latency"] = result["latency_s"]
        row[f"{strategy_name}_tokens"] = result["output_tokens"]

        completed += 1
        print(f"  [{completed:2d}/{total}] {item['id']:4s} | {strategy_name:22s} | "
              f"{result['latency_s']:5.2f}s | {result['output_tokens']:3d} tok")

        # Small delay to respect Groq rate limits
        time.sleep(0.3)

    results.append(row)

print("=" * 80)
print(f"All {total} inferences complete!")
print()

Running 45 inferences across 15 questions × 3 strategies...
  [ 1/45] c01  | A — Zero-Shot Direct   |  0.80s | 199 tok
  [ 2/45] c01  | B — Zero-Shot Role     |  0.51s | 108 tok
  [ 3/45] c01  | C — Few-Shot           |  0.42s | 155 tok
  [ 4/45] c02  | A — Zero-Shot Direct   |  0.54s | 182 tok
  [ 5/45] c02  | B — Zero-Shot Role     |  0.46s | 113 tok
  [ 6/45] c02  | C — Few-Shot           |  0.35s |  92 tok
  [ 7/45] c03  | A — Zero-Shot Direct   |  0.52s | 200 tok
  [ 8/45] c03  | B — Zero-Shot Role     |  0.35s | 100 tok
  [ 9/45] c03  | C — Few-Shot           |  0.45s | 135 tok
  [10/45] c04  | A — Zero-Shot Direct   |  0.56s | 188 tok
  [11/45] c04  | B — Zero-Shot Role     |  0.44s | 105 tok
  [12/45] c04  | C — Few-Shot           |  0.40s |  72 tok
  [13/45] c05  | A — Zero-Shot Direct   |  0.39s | 148 tok
  [14/45] c05  | B — Zero-Shot Role     |  0.29s |  65 tok
  [15/45] c05  | C — Few-Shot           |  0.47s | 134 tok
  [16/45] c06  | A — Zero-Shot Direct   |  0.31s |  92 

In [9]:
import csv

# ═══════════════════════════════════════════════════════════════
# 6. Export Raw Outputs to CSV
# ═══════════════════════════════════════════════════════════════

csv_path = "baseline_raw_outputs.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print(f"""Raw outputs saved to {csv_path} → Open this file in your spreadsheet to score each output manually (1-5)""")
print()

Raw outputs saved to baseline_raw_outputs.csv → Open this file in your spreadsheet to score each output manually (1-5)



In [10]:
import evaluate

# ═══════════════════════════════════════════════════════════════
# 7. Automated Scoring with ROUGE-L
# ═══════════════════════════════════════════════════════════════

rouge = evaluate.load("rouge")

print("=" * 80)
print("ROUGE-L SCORES — AUTOMATED")
print("=" * 80)

for strategy_name in STRATEGIES:
    predictions = [r[f"{strategy_name}_output"] for r in results]
    references = [r["expected"] for r in results]

    scores = rouge.compute(predictions=predictions, references=references)

    print(f"\n{strategy_name}")
    print(f"  ROUGE-1: {scores['rouge1']:.4f}")
    print(f"  ROUGE-2: {scores['rouge2']:.4f}")
    print(f"  ROUGE-L: {scores['rougeL']:.4f}")

    # Per-category breakdown
    for cat in ["Contracts", "NDAs", "Intellectual Property"]:
        cat_items = [r for r in results if r["category"] == cat]
        cat_predictions = [r[f"{strategy_name}_output"] for r in cat_items]
        cat_refs = [r["expected"] for r in cat_items]
        cat_scores = rouge.compute(predictions=cat_predictions, references=cat_refs)
        print(f"    {cat:25s}: ROUGE-L = {cat_scores['rougeL']:.4f}")

print()


ROUGE-L SCORES — AUTOMATED

A — Zero-Shot Direct
  ROUGE-1: 0.1794
  ROUGE-2: 0.0643
  ROUGE-L: 0.1378
    Contracts                : ROUGE-L = 0.1503
    NDAs                     : ROUGE-L = 0.1172
    Intellectual Property    : ROUGE-L = 0.1373

B — Zero-Shot Role
  ROUGE-1: 0.2704
  ROUGE-2: 0.1126
  ROUGE-L: 0.2222
    Contracts                : ROUGE-L = 0.2437
    NDAs                     : ROUGE-L = 0.1568
    Intellectual Property    : ROUGE-L = 0.2452

C — Few-Shot
  ROUGE-1: 0.2576
  ROUGE-2: 0.1067
  ROUGE-L: 0.2133
    Contracts                : ROUGE-L = 0.2280
    NDAs                     : ROUGE-L = 0.1756
    Intellectual Property    : ROUGE-L = 0.2155



In [11]:
# ═══════════════════════════════════════════════════════════════
# 8. Summary Table
# ═══════════════════════════════════════════════════════════════

print("=" * 80)
print("BASELINE SUMMARY")
print("=" * 80)

# Aggregate metrics by strategy
summary = {}
for strategy_name in STRATEGIES:
    predictions = [r[f"{strategy_name}_output"] for r in results]
    references = [r["expected"] for r in results]
    scores = rouge.compute(predictions=predictions, references=references)

    avg_latency = sum(r[f"{strategy_name}_latency"] for r in results) / len(results)
    avg_tokens = sum(r[f"{strategy_name}_tokens"] for r in results) / len(results)

    summary[strategy_name] = {
        "rouge_l": scores["rougeL"],
        "rouge_1": scores["rouge1"],
        "rouge_2": scores["rouge2"],
        "avg_latency_s": round(avg_latency, 2),
        "avg_output_tokens": round(avg_tokens, 1),
    }

# Print table
print(
    f"\n{'Strategy':<25s} {'ROUGE-L':<10s} {'ROUGE-1':<10s} {'ROUGE-2':<10s} "
    f"{'Latency':<10s} {'Tokens':<10s}"
)
print("-" * 75)
for strategy_name, metrics in summary.items():
    print(
        f"{strategy_name:<25s} {metrics['rouge_l']:<10.4f} {metrics['rouge_1']:<10.4f} "
        f"{metrics['rouge_2']:<10.4f} {metrics['avg_latency_s']:<10.2f} "
        f"{metrics['avg_output_tokens']:<10.1f}"
    )

# Gap A → C — the LoRA opportunity metric
gap_rouge_l = (
    summary["C — Few-Shot"]["rouge_l"] - summary["A — Zero-Shot Direct"]["rouge_l"]
)
gap_rouge_1 = (
    summary["C — Few-Shot"]["rouge_1"] - summary["A — Zero-Shot Direct"]["rouge_1"]
)
gap_rouge_2 = (
    summary["C — Few-Shot"]["rouge_2"] - summary["A — Zero-Shot Direct"]["rouge_2"]
)

print(
    f"\n{'Gap A → C (LoRA opportunity):':<25s} ROUGE-L: {gap_rouge_l:+.4f}  "
    f"ROUGE-1: {gap_rouge_1:+.4f}  ROUGE-2: {gap_rouge_2:+.4f}"
)
print()

BASELINE SUMMARY



Strategy                  ROUGE-L    ROUGE-1    ROUGE-2    Latency    Tokens    
---------------------------------------------------------------------------
A — Zero-Shot Direct      0.1378     0.1794     0.0643     1.21       194.6     
B — Zero-Shot Role        0.2222     0.2704     0.1126     1.09       103.7     
C — Few-Shot              0.2133     0.2576     0.1067     1.09       122.7     

Gap A → C (LoRA opportunity): ROUGE-L: +0.0754  ROUGE-1: +0.0782  ROUGE-2: +0.0424



In [12]:
import datetime
import json

# ═══════════════════════════════════════════════════════════════
# 9. Save Summary to JSON
# ═══════════════════════════════════════════════════════════════

output = {
    "timestamp": datetime.datetime.now().isoformat(),
    "model": MODEL_ID,
    "temperature": TEMPERATURE,
    "max_tokens": MAX_TOKENS,
    "dataset_size": len(DATASET),
    "strategies": summary,
    "gaps": {
        "a_to_c_rouge_l": round(gap_rouge_l, 4),
        "a_to_c_rouge_1": round(gap_rouge_1, 4),
        "a_to_c_rouge_2": round(gap_rouge_2, 4),
    },
}

with open("baseline_summary.json", "w") as f:
    json.dump(output, f, indent=2)

print("Summary saved to baseline_summary.json")
print(json.dumps(output, indent=2))

Summary saved to baseline_summary.json
{
  "timestamp": "2026-07-14T09:39:36.283109",
  "model": "llama-3.1-8b-instant",
  "temperature": 0.1,
  "max_tokens": 256,
  "dataset_size": 15,
  "strategies": {
    "A \u2014 Zero-Shot Direct": {
      "rouge_l": 0.13782062253862332,
      "rouge_1": 0.17938400253731374,
      "rouge_2": 0.06433311374866993,
      "avg_latency_s": 1.21,
      "avg_output_tokens": 194.6
    },
    "B \u2014 Zero-Shot Role": {
      "rouge_l": 0.22222666422378148,
      "rouge_1": 0.27037307237968433,
      "rouge_2": 0.11263975656350231,
      "avg_latency_s": 1.09,
      "avg_output_tokens": 103.7
    },
    "C \u2014 Few-Shot": {
      "rouge_l": 0.2132594447149999,
      "rouge_1": 0.25762735977893614,
      "rouge_2": 0.10670360023796191,
      "avg_latency_s": 1.09,
      "avg_output_tokens": 122.7
    }
  },
  "gaps": {
    "a_to_c_rouge_l": 0.0754,
    "a_to_c_rouge_1": 0.0782,
    "a_to_c_rouge_2": 0.0424
  }
}


In [13]:
from bert_score import score as bertscore

# Compute BERTScore for each strategy
for strat_name in STRATEGIES:
    predictions = [r[f"{strat_name}_output"] for r in results]
    references = [r["expected"] for r in results]
    
    P, R, F1 = bertscore(predictions, references, lang="en", verbose=False)
    
    print(f"\n{strat_name}")
    print(f"  BERTScore Precision: {P.mean().item():.4f}")
    print(f"  BERTScore Recall:    {R.mean().item():.4f}")
    print(f"  BERTScore F1:        {F1.mean().item():.4f}")

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 1041.02it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



A — Zero-Shot Direct
  BERTScore Precision: 0.8261
  BERTScore Recall:    0.8940
  BERTScore F1:        0.8586


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2049.94it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



B — Zero-Shot Role
  BERTScore Precision: 0.8591
  BERTScore Recall:    0.9139
  BERTScore F1:        0.8855


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2519.94it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



C — Few-Shot
  BERTScore Precision: 0.8515
  BERTScore Recall:    0.8990
  BERTScore F1:        0.8744
